In [1]:
import pandas as pd
import statsmodels.formula.api as smf
import numpy as np
import statsmodels.api as sm
from scipy.stats import skew

In [2]:
# Load CSV Files
df = pd.read_csv("homework_2.1.csv")
df2 = pd.read_csv("homework_2.2.csv")

In [ ]:
df.head

In [ ]:
# Reshape data to long format 

df_long = df.melt(
    id_vars = 'time',
    value_vars = ['G1', 'G2', 'G3'],
    var_name = 'group',
    value_name = 'outcome'
)

df_long.head()

In [ ]:
# Run the fixed effects regression

model = smf.ols(
    'outcome ~ time + C(group)',
    data = df_long
).fit()

model.summary()
model.params

In [ ]:
# Check dataset 2
df2.head()


In [ ]:
# Subtract the outcomes of the treated vs untreated population

treated_mean = df2[df2['X'] == 1]['Y'].mean()
untreated_mean = df2[df2['X'] == 0]['Y'].mean()

effect1 = treated_mean - untreated_mean

# Print the mean effect of this
print(effect1)

In [ ]:
# Bootstrap the simple effect 

np.random.seed(42)

effects = []

for i in range(10000):
    sample = df2.sample(n=len(df2), replace = True)

    effect = (
        sample[sample['X'] == 1]['Y'].mean() - 
        sample[sample['X'] == 0]['Y'].mean()
    )

    effects.append(effect)

# Take the variance of those bootstrapped effects

bootstrap_variance = np.var(effects, ddof=1)

print(bootstrap_variance)

In [3]:
# Bootstrap many samples, fit a linear regression with intercept, 
# save the treatment effect coefficient, build a list of those coefficients,
# compute the skewness of that list

# Bootstrap the Regression Coefficient

effects2 = []

for i in range(10000):

    sample = df2.sample(
        n = len(df2),
        replace = True
    )

    X = sm.add_constant(sample['X']) # adds intercept
    y = sample['Y']

    model = sm.OLS(y, X).fit()

    effects2.append(
        model.params['X']
    )

In [4]:
# Compute skewness

effect_skewness = skew(effects2)

print(effect_skewness)

0.05204849421708369
